# 05 - Model Training (Forecast Engine)

## Objective
Train and tune a machine learning pipeline that predicts **next-day return** for each StockMetrics ticker.

This trained model will later be used by the dashboard to support simple forecast-style scenario ranges
(Optimistic / Realistic / Pessimistic), helping beginners understand uncertainty without needing complex finance.

## Inputs
- Feature dataset: `data/processed/<version>/features_<version>_latest.csv`
- Version label (e.g. `v1`)

## Outputs
- Saved trained pipeline: `models/stock_forecast_model_<version>.pkl`
- Training report (metrics + best params): `outputs/<version>/reports/model_training_report_<version>.json`
- Console summary: train/test metrics and best hyperparameters

## CRISP-DM Stage
Modelling

In [ ]:
# Make the project root importable (so `import src...` works in notebooks)
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()  # notebooks live in jupyter_notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root added to sys.path:", PROJECT_ROOT)

In [ ]:
import json
import pandas as pd

from src.config import DEFAULT_VERSION, get_paths
from src.features import load_features_latest
from src.modelling import train_and_tune, save_model

import numpy as np
np.random.seed(42)

In [ ]:
VERSION = DEFAULT_VERSION
paths = get_paths(VERSION)

MODELS_DIR = paths.models_dir
REPORTS_DIR = paths.outputs_dir / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Version:", VERSION)
print("Models dir:", MODELS_DIR)
print("Reports dir:", REPORTS_DIR)

In [ ]:
feat_df = load_features_latest(paths.processed_dir, VERSION)

print("Loaded features shape:", feat_df.shape)
print("Tickers:", sorted(feat_df["Ticker"].unique().tolist()))
print("Date range:", feat_df["Date"].min().date(), "to", feat_df["Date"].max().date())

display(feat_df.head(5))
display(feat_df.groupby("Ticker").head(2))

In [ ]:
required_cols = {"Date", "Ticker", "return_1d"}
missing = required_cols - set(feat_df.columns)
if missing:
    raise KeyError(f"Missing required columns in features dataset: {missing}")

print("Target column present:", "return_1d" in feat_df.columns)
print("Non-null target rows:", feat_df["return_1d"].notna().sum())

## ML Business Case (Training Notebook)

### Predictive Task
This project uses **supervised learning (regression)** to predict a ticker’s **next-day return** based on engineered historical features.

### Why this matters for StockMetrics
StockMetrics aims to reduce analysis paralysis for beginner investors by:

- highlighting volatility and risk behaviour,
- presenting outcomes as scenario forecast ranges (optimistic / realistic / pessimistic),
- keeping the focus on a small, curated set of well-known tickers.

The predicted daily return acts as a building block to simulate price paths over longer horizons and generate scenario ranges.

### Success Criteria
- Primary metric: **R²** on a chronological (time-aware) test split.
- Secondary metrics: **MAE** and **RMSE**.

If test-set R² is close to 0 or negative (which is common in financial return prediction),
the model will still be useful for illustrating uncertainty but forecasts must be framed
as probabilistic ranges rather than precise predictions.

In [ ]:
# Basic checks before fitting any models
print("Rows:", len(feat_df))
print("Columns:", len(feat_df.columns))
print("Tickers:", feat_df["Ticker"].nunique())
print("Date min/max:", feat_df["Date"].min(), "->", feat_df["Date"].max())
print("Any NA in target:", feat_df["return_1d"].isna().any())

# Small random subset to make sure the training pipeline runs quickly
# (does not modify the saved dataset)
smoke_df = feat_df.sample(3000, random_state=42).sort_values(["Date", "Ticker"])
print("Smoke test rows:", len(smoke_df))

In [ ]:
print("Rows for training (smoke test):", len(smoke_df))

result = train_and_tune(smoke_df, test_size=0.2, fast=True)

print("\nBest Params:")
for k, v in result.best_params.items():
    print(f"- {k}: {v}")

print("\nTrain Metrics:")
for k, v in result.metrics_train.items():
    print(f"- {k}: {v:.6f}")

print("\nTest Metrics:")
for k, v in result.metrics_test.items():
    print(f"- {k}: {v:.6f}")

In [ ]:
# Medium-size test (faster than full dataset, bigger than smoke test)
medium_df = (
    feat_df.sample(10000, random_state=42)
    .sort_values(["Date", "Ticker"])
    .reset_index(drop=True)
)

print("Medium test rows:", len(medium_df))

In [ ]:
print("Rows for training (medium test):", len(medium_df))

result = train_and_tune(medium_df, test_size=0.2, fast=True)

print("\nBest Params:")
for k, v in result.best_params.items():
    print(f"- {k}: {v}")

print("\nTrain Metrics:")
for k, v in result.metrics_train.items():
    print(f"- {k}: {v:.6f}")

print("\nTest Metrics:")
for k, v in result.metrics_test.items():
    print(f"- {k}: {v:.6f}")

In [ ]:
# Full dataset run (still using fast grid for now)
print("Rows for training:", len(feat_df))

result = train_and_tune(feat_df, test_size=0.2, fast=True)

print("\nBest Params:")
for k, v in result.best_params.items():
    print(f"- {k}: {v}")

print("\nTrain Metrics:")
for k, v in result.metrics_train.items():
    print(f"- {k}: {v:.6f}")

print("\nTest Metrics:")
for k, v in result.metrics_test.items():
    print(f"- {k}: {v:.6f}")